In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from featureEngineer import engineer_features
import warnings

from pathlib import Path
import sys
import os
from datetime import datetime
from dataScraper import *

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

# from src.utils.team_info import nameDict
warnings.filterwarnings("ignore")

In [7]:
pd.set_option('display.max_columns', None)

s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')

In [8]:
df = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])
df = df.sort_values(["PLAYER_ID", "GAME_DATE"]).reset_index(drop=True)
 
# Derived features
df["PTS_PER_MIN"] = df["PTS"] / df["MIN"].replace(0, np.nan)
df["IS_HOME"]     = df["MATCHUP"].str.contains("vs\\.").astype(int)
df["SPREAD_PROXY"] = df["TEAM_PLUS_MINUS"]  # actual outcome; use betting spread at prediction time

df.head()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,PTS_PER_MIN,IS_HOME,SPREAD_PROXY
0,18066,2025-26,2544,LeBron James,LeBron,1610612747,LAL,Los Angeles Lakers,22500253,2025-11-18,LAL vs. UTA,W,29.616667,4,7,0.571,2,3,0.667,1,4,0.250,1,2,3,12,1,1,0,0,0,3,11,1,34.6,1,0,30.0,1,29:37,1,133.8,133.8,133.8,123.1,128.4,128.4,10.6,5.5,5.5,0.414,12.0,54.5,0.040,0.083,0.061,4.5,4.6,0.714,0.628,0.135,0.135,109.30,106.97,89.14,106.97,0.110,65,4.0,7.0,50,84,0.595,11,32,0.344,29,40,0.725,11,30,41,31,17.0,9,1,2,21,30,140,14.0,130.1,130.8,114.6,118.9,15.5,12.0,0.620,1.82,20.4,0.350,0.727,0.548,0.159,0.661,0.689,108.8,106.5,88.75,107,0.550,1610612762,UTA,Utah Jazz,48,92,0.522,13,45,0.289,17,18,0.944,8,25,33,33,18.0,13,2,1,30,21,126,-14.0,114.6,118.9,130.1,130.8,-15.5,-12.0,0.688,1.83,21.9,0.273,0.650,0.452,0.170,0.592,0.631,108.8,106.5,88.75,106,0.450,F,0.371412,1,14.0
1,17269,2025-26,2544,LeBron James,LeBron,1610612747,LAL,Los Angeles Lakers,22500282,2025-11-23,LAL @ UTA,W,34.321667,8,18,0.444,0,4,0.000,1,2,0.500,0,6,6,8,2,1,0,0,2,2,17,-14,37.2,0,0,33.0,1,34:19,1,96.8,97.3,97.3,115.8,116.2,116.2,-19.0,-18.9,-18.9,0.444,4.0,27.6,0.000,0.167,0.083,6.9,6.9,0.444,0.450,0.266,0.266,103.94,103.49,86.24,103.49,0.124,74,8.0,18.0,38,86,0.442,10,38,0.263,22,27,0.815,5,40,45,23,14.0,7,4,2,18,21,108,2.0,101.0,105.9,100.1,102.9,1.0,3.0,0.605,1.64,17.2,0.204,0.719,0.481,0.137,0.500,0.552,106.4,102.5,85.42,102,0.528,1610612762,UTA,Utah Jazz,40,94,0.426,13,40,0.325,13,18,0.722,10,35,45,27,14.0,7,2,4,21,18,106,-2.0,100.1,102.9,101.0,105.9,-1.0,-3.0,0.675,1.93,18.6,0.281,0.796,0.519,0.136,0.495,0.520,106.4,102.5,85.42,103,0.472,F,0.495314,0,2.0
2,16951,2025-26,2544,LeBron James,LeBron,1610612747,LAL,Los Angeles Lakers,22500059,2025-11-25,LAL vs. LAC,W,32.316667,9,15,0.600,2,5,0.400,5,6,0.833,1,5,6,6,3,1,1,0,3,4,25,18,44.2,0,0,43.0,1,32:19,1,148.7,143.3,143.3,119.3,123.8,123.8,29.4,19.5,19.5,0.250,2.0,23.1,0.038,0.185,0.113,11.5,11.3,0.667,0.709,0.250,0.281,96.51,96.54,80.45,96.54,0.152,67,9.0,15.0,50,87,0.575,13,29,0.448,22,26,0.846,12,26,38,29,11.0,9,4,2,22,22,135,17.0,138.5,139.2,117.3,124.2,21.2,15.0,0.580,2.64,20.1,0.429,0.675,0.549,0.113,0.649,0.686,99.0,96.0,80.00,97,0.582,1610612746,LAC,LA Clippers,42,81,0.519,13,35,0.371,21,24,0.875,7,22,29,24,16.0,7,2,4,22,22,118,-17.0,117.3,124.2,138.5,139.2,-21.2,-15.0,0.571,1.50,18.2,0.325,0.571,0.451,0.168,0.599,0.644,99.0,96.0,80.00,95,0.418,F,0.773595,1,17.0
3,16601,2025-26,2544,LeBron

In [ ]:
def rolling_player(df, col, windows=[5,10], min_periods=1):
    """Per-player rolling means (shifted to avoid leakage)."""
    out = {}
    for w in windows:
        out[f"{col}_roll{w}"] = (
            df.groupby("PLAYER_ID")[col]
            .transform(lambda x: x.shift(1).rolling(w, min_periods=min_periods).mean())
        )
    return out
 
for col in ["MIN", "PTS_PER_MIN", "USG_PCT", "PACE"]:
    for k, v in rolling_player(df, col).items():
        df[k] = v
 
# Exponential weighted mean (λ decay, equivalent to span≈7 games)
df["MIN_ewm"]         = df.groupby("PLAYER_ID")["MIN"].transform(lambda x: x.shift(1).ewm(span=7).mean())
df["PTS_PER_MIN_ewm"] = df.groupby("PLAYER_ID")["PTS_PER_MIN"].transform(lambda x: x.shift(1).ewm(span=7).mean())
 
# Games played (sample size tracker)
df["GP"] = df.groupby("PLAYER_ID").cumcount()  # 0-indexed; GP=0 means first game
 
# Blowout risk proxy: rolling team point differential variance
df["BLOWOUT_RISK"] = df.groupby("PLAYER_ID")["TEAM_PLUS_MINUS"].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).std()
)
df['STARTER_FLAG'] = df['START_POSITION'].notna().astype(int)
df['TEAM_MIN_RANK_L10'] = (df.groupby(['TEAM_ID','GAME_DATE'])['MIN_roll10'].rank(ascending=False, method='dense'))
df.head()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,PTS_PER_MIN,IS_HOME,SPREAD_PROXY,MIN_roll5,MIN_roll10,PTS_PER_MIN_roll5,PTS_PER_MIN_roll10,USG_PCT_roll5,USG_PCT_roll10,PACE_roll5,PACE_roll10,MIN_ewm,PTS_PER_MIN_ewm,GP,BLOWOUT_RISK,STARTER_FLAG,TEAM_MIN_RANK_L10,prior_mean,prior_std,BAYES_PTS_PER_MIN,MIN_roll3,PTS_PER_MIN_roll3,USG_PCT_roll3,PACE_roll3
0,18066,2025-26,2544,LeBron James,LeBron,1610612747,LAL,Los Angeles Lakers,22500253,2025-11-18,LAL vs. UTA,W,29.616667,4,7,0.571,2,3,0.667,1,4,0.250,1,2,3,12,1,1,0,0,0,3,11,1,34.6,1,0,30.0,1,29:37,1,133.8,133.8,133.8,123.1,128.4,128.4,10.6,5.5,5.5,0.414,12.0,54.5,0.040,0.083,0.061,4.5,4.6,0.714,0.628,0.135,0.135,109.30,106.97,89.14,106.97,0.110,65,4.0,7.0,50,84,0.595,11,32,0.344,29,40,0.725,11,30,41,31,17.0,9,1,2,21,30,140,14.0,130.1,130.8,114.6,118.9,15.5,12.0,0.620,1.82,20.4,0.350,0.727,0.548,0.159,0.661,0.689,108.8,106.5,88.75,107,0.550,1610612762,UTA,Utah Jazz,48,92,0.522,13,45,0.289,17,18,0.944,8,25,33,33,18.0,13,2,1,30,21,126,-14.0,114.6,118.9,130.1,130.8,-15.5,-12.0,0.688,1.83,21.9,0.273,0.650,0.452,0.170,0.592,0.631,108.8,106.5,88.75,106,0.450,F,0.371412,1,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,1,NaN,0.509133,0.249236,0.509133,NaN,NaN,NaN,NaN
1,17269,2025-26,2544,LeBron James,LeBron,1610612747,LAL,Los Angeles Lakers,22500282,2025-11-23,LAL @ UTA,W,34.321667,8,18,0.444,0,4,0.000,1,2,0.500,0,6,6,8,2,1,0,0,2,2,17,-14,37.2,0,0,33.0,1,34:19,1,96.8,97.3,97.3,115.8,116.2,116.2,-19.0,-18.9,-18.9,0.444,4.0,27.6,0.000,0.167,0.083,6.9,6.9,0.444,0.450,0.266,0.266,103.94,103.49,86.24,103.49,0.124,74,8.0,18.0,38,86,0.442,10,38,0.263,22,27,0.815,5,40,45,23,14.0,7,4,2,18,21,108,2.0,101.0,105.9,100.1,102.9,1.0,3.0,0.605,1.64,17.2,0.204,0.719,0.481,0.137,0.500,0.552,106.4,102.5,85.42,102,0.528,1610612762,UTA,Utah Jazz,40,94,0.426,13,40,0.325,13,18,0.722,10,35,45,27,14.0,7,2,4,21,18,106,-2.0,100.1,102.9,101.0,105.9,-1.0,-3.0,0.675,1.93,18.6,0.281,0.796,0.519,0.136,0.495,0.520,106.4,102.5,85.42,103,0.472,F,0.495314,0,2.0,29.616667,29.616667,0.371412,0.371412,0.13500,0.13500,106.970000,106.970000,29.616667,0.371412,1,NaN,1,4.0,0.509133,0.249236,0.440273,29.616667,0.371412,0.135000,106.970000
2,16951,2025-26,2544,LeBron James,LeBron,1610612747,LAL,Los Angeles Lakers,22500059,2025-11-25,LAL vs. LAC,W,32.316667,9,15,0.600,2,5,0.400,5,6,0.833,1,5,6,6,3,1,1,0,3,4,25,18,44.2,0,0,43.0,1,32:19,1,148.7,143.3,143.3,119.3,123.8,123.8,29.4,19.5,19.5,0.250,2.0,23

In [14]:
# Drop stale prior columns so re-running this cell does not merge into prior_mean_x / prior_mean_y
_prior_stale = [
    c
    for c in df.columns
    if c in ("prior_mean", "prior_std")
    or c.startswith("prior_mean_")
    or c.startswith("prior_std_")
]
if _prior_stale:
    df = df.drop(columns=_prior_stale)

league_prior = (
    df.groupby("STARTER_FLAG")["PTS_PER_MIN"]
    .agg(prior_mean="mean", prior_std="std")
    .reset_index()
)
df = df.merge(league_prior, on="STARTER_FLAG", how="left")

# Conjugate normal–normal posterior mean (vectorized)
prior_m = df["prior_mean"]
prior_v = (df["prior_std"] ** 2).clip(lower=1e-12)

obs_mean = df["PTS_PER_MIN_ewm"].where(df["PTS_PER_MIN_ewm"].notna(), prior_m)
obs_n = df["GP"].clip(upper=20)
obs_var = prior_v

prior_prec = 1 / prior_v
obs_prec = obs_n / obs_var
posterior_mean = (prior_prec * prior_m + obs_prec * obs_mean) / (prior_prec + obs_prec)

df["BAYES_PTS_PER_MIN"] = np.where(obs_n.to_numpy() == 0, prior_m.to_numpy(), posterior_mean.to_numpy())
df.head()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,PTS_PER_MIN,IS_HOME,SPREAD_PROXY,MIN_roll5,MIN_roll10,PTS_PER_MIN_roll5,PTS_PER_MIN_roll10,USG_PCT_roll5,USG_PCT_roll10,PACE_roll5,PACE_roll10,MIN_ewm,PTS_PER_MIN_ewm,GP,BLOWOUT_RISK,STARTER_FLAG,TEAM_MIN_RANK_L10,BAYES_PTS_PER_MIN,MIN_roll3,PTS_PER_MIN_roll3,USG_PCT_roll3,PACE_roll3,prior_mean,prior_std
0,18066,2025-26,2544,LeBron James,LeBron,1610612747,LAL,Los Angeles Lakers,22500253,2025-11-18,LAL vs. UTA,W,29.616667,4,7,0.571,2,3,0.667,1,4,0.250,1,2,3,12,1,1,0,0,0,3,11,1,34.6,1,0,30.0,1,29:37,1,133.8,133.8,133.8,123.1,128.4,128.4,10.6,5.5,5.5,0.414,12.0,54.5,0.040,0.083,0.061,4.5,4.6,0.714,0.628,0.135,0.135,109.30,106.97,89.14,106.97,0.110,65,4.0,7.0,50,84,0.595,11,32,0.344,29,40,0.725,11,30,41,31,17.0,9,1,2,21,30,140,14.0,130.1,130.8,114.6,118.9,15.5,12.0,0.620,1.82,20.4,0.350,0.727,0.548,0.159,0.661,0.689,108.8,106.5,88.75,107,0.550,1610612762,UTA,Utah Jazz,48,92,0.522,13,45,0.289,17,18,0.944,8,25,33,33,18.0,13,2,1,30,21,126,-14.0,114.6,118.9,130.1,130.8,-15.5,-12.0,0.688,1.83,21.9,0.273,0.650,0.452,0.170,0.592,0.631,108.8,106.5,88.75,106,0.450,F,0.371412,1,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,1,NaN,0.509133,NaN,NaN,NaN,NaN,0.509133,0.249236
1,17269,2025-26,2544,LeBron James,LeBron,1610612747,LAL,Los Angeles Lakers,22500282,2025-11-23,LAL @ UTA,W,34.321667,8,18,0.444,0,4,0.000,1,2,0.500,0,6,6,8,2,1,0,0,2,2,17,-14,37.2,0,0,33.0,1,34:19,1,96.8,97.3,97.3,115.8,116.2,116.2,-19.0,-18.9,-18.9,0.444,4.0,27.6,0.000,0.167,0.083,6.9,6.9,0.444,0.450,0.266,0.266,103.94,103.49,86.24,103.49,0.124,74,8.0,18.0,38,86,0.442,10,38,0.263,22,27,0.815,5,40,45,23,14.0,7,4,2,18,21,108,2.0,101.0,105.9,100.1,102.9,1.0,3.0,0.605,1.64,17.2,0.204,0.719,0.481,0.137,0.500,0.552,106.4,102.5,85.42,102,0.528,1610612762,UTA,Utah Jazz,40,94,0.426,13,40,0.325,13,18,0.722,10,35,45,27,14.0,7,2,4,21,18,106,-2.0,100.1,102.9,101.0,105.9,-1.0,-3.0,0.675,1.93,18.6,0.281,0.796,0.519,0.136,0.495,0.520,106.4,102.5,85.42,103,0.472,F,0.495314,0,2.0,29.616667,29.616667,0.371412,0.371412,0.13500,0.13500,106.970000,106.970000,29.616667,0.371412,1,NaN,1,4.0,0.440273,29.616667,0.371412,0.135000,106.970000,0.509133,0.249236
2,16951,2025-26,2544,LeBron James,LeBron,1610612747,LAL,Los Angeles Lakers,22500059,2025-11-25,LAL vs. LAC,W,32.316667,9,15,0.600,2,5,0.400,5,6,0.833,1,5,6,6,3,1,1,0,3,4,25,18,44.2,0,0,43.0,1,32:19,1,148.7,143.3,143.3,119.3,123.8,123.8,29.4,19.5,19.5,0.250,2.0,23

In [24]:
MIN_FEATURES = [
    "MIN_roll3", "MIN_roll5", "MIN_roll10",
    "MIN_ewm",
    "STARTER_FLAG",
    "BLOWOUT_RISK",
    "IS_HOME",
    "PACE_roll3",
    "GP",
    'TEAM_MIN_RANK_L10'
]
 
# Drop rows with NaN in features or target
min_df = df[MIN_FEATURES + ["MIN", "GAME_DATE"]].dropna()
 
# ── TIME-BASED SPLIT ──
split_date = min_df["GAME_DATE"].quantile(0.75)  # train on first 75% of season
train_mask = min_df["GAME_DATE"] <= split_date
val_mask   = ~train_mask
 
X_train_min = min_df[MIN_FEATURES][train_mask]
y_train_min = min_df["MIN"][train_mask]
X_val_min   = min_df[MIN_FEATURES][val_mask]
y_val_min   = min_df["MIN"][val_mask]
 
scaler_min = StandardScaler()
X_train_min_s = scaler_min.fit_transform(X_train_min)
X_val_min_s   = scaler_min.transform(X_val_min)
 
ridge_min = Ridge(alpha=10.0)
ridge_min.fit(X_train_min_s, y_train_min)
 
min_preds_val = ridge_min.predict(X_val_min_s)
min_mae = mean_absolute_error(y_val_min, min_preds_val)
print(f"[Minutes Model]  Validation MAE: {min_mae:.2f} min")

[Minutes Model]  Validation MAE: 4.66 min


In [16]:
PPM_FEATURES = [
    "BAYES_PTS_PER_MIN",
    "PTS_PER_MIN_roll3", "PTS_PER_MIN_roll5",
    "PTS_PER_MIN_ewm",
    "USG_PCT_roll3",
    "PACE_roll3",
    "IS_HOME",
    "STARTER_FLAG",
    "GP",
    # Opponent defense
    "OPP_DEF_RATING",
]
 
# Check column exists
PPM_FEATURES = [f for f in PPM_FEATURES if f in df.columns]
 
ppm_df = df[PPM_FEATURES + ["PTS_PER_MIN", "GAME_DATE"]].replace([np.inf, -np.inf], np.nan).dropna()
 
split_date_ppm = ppm_df["GAME_DATE"].quantile(0.75)
train_mask_ppm = ppm_df["GAME_DATE"] <= split_date_ppm
val_mask_ppm   = ~train_mask_ppm
 
X_train_ppm = ppm_df[PPM_FEATURES][train_mask_ppm]
y_train_ppm = ppm_df["PTS_PER_MIN"][train_mask_ppm]
X_val_ppm   = ppm_df[PPM_FEATURES][val_mask_ppm]
y_val_ppm   = ppm_df["PTS_PER_MIN"][val_mask_ppm]
 
scaler_ppm = StandardScaler()
X_train_ppm_s = scaler_ppm.fit_transform(X_train_ppm)
X_val_ppm_s   = scaler_ppm.transform(X_val_ppm)
 
ridge_ppm = Ridge(alpha=5.0)
ridge_ppm.fit(X_train_ppm_s, y_train_ppm)
 
ppm_preds_val = ridge_ppm.predict(X_val_ppm_s)
ppm_mae = mean_absolute_error(y_val_ppm, ppm_preds_val)
print(f"[Pts/Min Model]  Validation MAE: {ppm_mae:.4f} pts/min")

[Pts/Min Model]  Validation MAE: 0.1913 pts/min


In [17]:
# Reconstruct val set projection
val_common = ppm_df[val_mask_ppm].copy()
val_common["PRED_PTS_PER_MIN"] = ppm_preds_val
 
val_min_aligned = min_df[val_mask].copy()
val_min_aligned["PRED_MIN"] = min_preds_val
 
# Merge on index (both share original df index ordering)
# Simpler: recompute on shared rows
combined_idx = val_common.index.intersection(val_min_aligned.index)
val_combined = val_common.loc[combined_idx].copy()
val_combined["PRED_MIN"]         = val_min_aligned.loc[combined_idx, "PRED_MIN"]
val_combined["TRUE_PTS"]         = df.loc[combined_idx, "PTS"]
val_combined["PRED_PTS"]         = val_combined["PRED_MIN"] * val_combined["PRED_PTS_PER_MIN"]
 
pts_mae  = mean_absolute_error(val_combined["TRUE_PTS"], val_combined["PRED_PTS"])
pts_bias = (val_combined["PRED_PTS"] - val_combined["TRUE_PTS"]).mean()
print(f"[Points Proj]    Validation MAE:  {pts_mae:.2f} pts")
print(f"[Points Proj]    Bias (pred-true): {pts_bias:.2f} pts")
 

[Points Proj]    Validation MAE:  4.51 pts
[Points Proj]    Bias (pred-true): -0.14 pts


In [22]:
def project_player(player_name: str, opp_def_rating: float = None, spread: float = None):
    """
    Project points for a player's next game.
    opp_def_rating: opponent defensive rating (lower = tougher defense)
    spread: positive = player's team favored
    """
    player_rows = df[df["PLAYER_NAME"].str.lower() == player_name.lower()].copy()
    if player_rows.empty:
        # Try partial match
        mask = df["PLAYER_NAME"].str.lower().str.contains(player_name.lower())
        player_rows = df[mask].copy()
    if player_rows.empty:
        return f"Player '{player_name}' not found."
 
    latest = player_rows.sort_values("GAME_DATE").iloc[-1]
 
    # ── Minutes projection ──
    min_row = {f: latest.get(f, np.nan) for f in MIN_FEATURES}
    min_row_df = pd.DataFrame([min_row])
    min_row_scaled = scaler_min.transform(min_row_df.fillna(min_row_df.mean()))
    proj_min = float(ridge_min.predict(min_row_scaled)[0])
 
    # Blowout adjustment on minutes
    if spread is not None and abs(spread) > 15:
        proj_min *= 0.90  # -10% for likely blowout
 
    # ── Pts/Min projection ──
    ppm_row = {f: latest.get(f, np.nan) for f in PPM_FEATURES}
    if opp_def_rating is not None:
        ppm_row["OPP_DEF_RATING"] = opp_def_rating
    ppm_row_df = pd.DataFrame([ppm_row])
    ppm_row_scaled = scaler_ppm.transform(ppm_row_df.fillna(ppm_row_df.mean()))
    proj_ppm = float(ridge_ppm.predict(ppm_row_scaled)[0])
 
    proj_pts = proj_min * proj_ppm
 
    # ── Output ──
    print(f"\n{'─'*45}")
    print(f"  Player:          {latest['PLAYER_NAME']}")
    print(f"  Last game:       {latest['GAME_DATE'].date()}  {latest['MATCHUP']}  {latest['PTS']} pts in {latest['MIN']:.1f} min")
    print(f"  Recent avg min:  {latest.get('MIN_roll5', np.nan):.1f}  (last 5 games)")
    print(f"  Starter flag:    {'Yes' if latest.get('STARTER_FLAG') else 'No'}")
    print(f"  Bayesian ppm:    {latest.get('BAYES_PTS_PER_MIN', np.nan):.3f}")
    print(f"  ── Projections ──")
    print(f"  Projected MIN:   {proj_min:.1f}")
    print(f"  Projected ppm:   {proj_ppm:.3f}")
    print(f"  Projected PTS:   {proj_pts:.1f}")
    print(f"{'─'*45}\n")
    return proj_pts

print("\n=== SAMPLE PROJECTIONS (next game) ===")
for name in ["Shai Gilgeous-Alexander"]:
    project_player(name)


=== SAMPLE PROJECTIONS (next game) ===

─────────────────────────────────────────────
  Player:          Shai Gilgeous-Alexander
  Last game:       2026-03-18  OKC @ BKN  20 pts in 25.6 min
  Recent avg min:  36.4  (last 5 games)
  Starter flag:    Yes
  Bayesian ppm:    0.866
  ── Projections ──
  Projected MIN:   34.5
  Projected ppm:   0.796
  Projected PTS:   27.5
─────────────────────────────────────────────

